In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from pathlib import Path
from joblib import Parallel, delayed, dump
from IPython.display import display

from pd_estim_A.data.data_import import (
    load_data,
    load_ecb_1y_yield,
    fill_liabilities,
    drop_high_leverage_firms,
    prepare_nig_inputs,
)
from pd_estim_A.models.nig.gibbs_nig_afonso import (
    process_one_firm_bayesian_nig,
    summarize_chain_diagnostics,
    build_mcmc_diagnostics_df
)
from pd_estim_A.data.cds_df import get_cds_panel

In [13]:
# Data processing and df preparation
print(Path.cwd())

data_path = Path.cwd() / ".." / "data" / "raw"
output_path = Path.cwd() / ".." / "data" / "derived"

# raw accounting / equity data
ret_daily, bs, coverage = load_data(
    data_path / "Jan2025_Accenture_Dataset_ErasmusCase.xlsx",
    start_date="2012-01-01",
    end_date="2025-12-19",
    enforce_coverage=True,
    coverage_tol=0.995,
    liabilities_scale="auto",
    verbose=True,
)

# risk-free rate
df_rf = load_ecb_1y_yield(
    startPeriod="2010-01-01",
    endPeriod="2025-12-31",
    out_file=output_path / "ecb_yc_1y_aaa.xml",
    verify_ssl=True,
)

# calendar and debt interpolation
df_cal = ret_daily[["date"]].drop_duplicates().sort_values("date").reset_index(drop=True)
debt_daily = fill_liabilities(bs, df_cal)

# leverage filter
ret_filt, bs_filt, lev_by_firm, dropped = drop_high_leverage_firms(
    ret_daily,
    bs,
    df_calendar=df_cal,
    debt_daily=debt_daily,
    lev_threshold=8.0,
    lev_agg="median",
    verbose=True,
)

# keep debt panel aligned with surviving firms
keep = set(ret_filt["gvkey"].astype(str).unique())
debt_daily_filt = debt_daily[debt_daily["gvkey"].astype(str).isin(keep)].copy()

# structural-model input panel for NIG
nig_out = prepare_nig_inputs(
    ret_filt,
    bs_filt,
    df_rf,
    debt_daily=debt_daily_filt,
)

# robustly extract the main DataFrame
if isinstance(nig_out, pd.DataFrame):
    nig_raw = nig_out.copy()
elif isinstance(nig_out, tuple):
    df_candidates = [x for x in nig_out if isinstance(x, pd.DataFrame)]
    if len(df_candidates) == 0:
        raise TypeError(
            "prepare_nig_inputs returned a tuple, but none of its elements is a DataFrame."
        )
    nig_raw = df_candidates[0].copy()
else:
    raise TypeError(
        f"prepare_nig_inputs returned unsupported type: {type(nig_out)}"
    )


# CDS panel
cds = get_cds_panel(
    project_root=Path.cwd() / "..",
    save_csv=False,
    verbose=True,
)

# harmonize types
nig = nig_raw.copy()
nig["gvkey"] = nig["gvkey"].astype(str)
nig["date"] = pd.to_datetime(nig["date"])

cds["gvkey"] = cds["gvkey"].astype(str)
cds["date"] = pd.to_datetime(cds["date"])

# keep only firms that appear in both datasets
common_gv = sorted(set(nig["gvkey"].unique()) & set(cds["gvkey"].unique()))
nig = nig[nig["gvkey"].isin(common_gv)].copy()
cds = cds[cds["gvkey"].isin(common_gv)].copy()

# restrict CDS to structural-panel date range
dmin, dmax = nig["date"].min(), nig["date"].max()
cds = cds[(cds["date"] >= dmin) & (cds["date"] <= dmax)].copy()

# merge CDS backward onto structural panel
nig = nig.sort_values(["date", "gvkey"]).reset_index(drop=True)
cds = cds.sort_values(["date", "gvkey"]).reset_index(drop=True)

merged_cds = pd.merge_asof(
    nig,
    cds,
    on="date",
    by="gvkey",
    direction="backward",
    allow_exact_matches=True,
)

# drop rows still missing CDS
nig_df = merged_cds.dropna(subset=["cds"]).reset_index(drop=True)

# normalize key column names to match the Bayesian-NIG module defaults
rename_map = {}
if "E" in nig_df.columns and "market_cap" not in nig_df.columns:
    rename_map["E"] = "market_cap"
if "B" in nig_df.columns and "debt_face" not in nig_df.columns:
    rename_map["B"] = "debt_face"
if "r" in nig_df.columns and "rf" not in nig_df.columns:
    rename_map["r"] = "rf"

nig_df = nig_df.rename(columns=rename_map)

print("firms after intersection:", nig_df["gvkey"].nunique())
print("rows after merge:", len(nig_df))
print("date range:", nig_df["date"].min(), "→", nig_df["date"].max())
print("columns available:", sorted(nig_df.columns))


c:\Users\afons\OneDrive\Desktop\ESE\FCS\Merton_NIGbayesian\notebooks_test
[load_data] Firms (ret_daily): 46
[load_data] Date range (ret_daily): 2012-01-03 .. 2025-12-19
[load_data] Coverage min/median/max: 0.999 / 1.000 / 1.000
[load_data] liabilities_scale_used: 1e+06
[load_data] QA mcap_reported<=0 rows (raw windowed mkt): 62
Data has been written to c:\Users\afons\OneDrive\Desktop\ESE\FCS\Merton_NIGbayesian\notebooks_test\..\data\derived\ecb_yc_1y_aaa.xml
[drop_high_leverage_firms] agg=median, threshold=8.0
[drop_high_leverage_firms] firms before: 46 | after: 36
[drop_high_leverage_firms] dropped firms: 10


KeyboardInterrupt: 

In [ ]:
# load precomputed frequentist NIG results for Bayesian initialization
# source file is a WEEKLY OOS panel, so we collapse it to one row per firm-window

freq_file = output_path / "nig_freq.csv"

if not freq_file.exists():
    raise FileNotFoundError(f"Could not find frequentist NIG file: {freq_file}")

freq_raw = pd.read_csv(freq_file)

# harmonize basic types
freq_raw["gvkey"] = freq_raw["gvkey"].astype(str)

date_cols = [
    "date",
    "window_train_start",
    "window_train_end",
    "window_oos_start",
    "window_oos_end",
]
for c in date_cols:
    if c in freq_raw.columns:
        freq_raw[c] = pd.to_datetime(freq_raw[c], errors="coerce")

num_cols = ["window_idx", "alpha", "beta1", "delta", "beta0"]
for c in num_cols:
    if c in freq_raw.columns:
        freq_raw[c] = pd.to_numeric(freq_raw[c], errors="coerce")

# keep only successful frequentist rows
if "ok" in freq_raw.columns:
    ok_mask = freq_raw["ok"].astype(str).str.lower().isin(["true", "1", "yes"])
    freq_raw = freq_raw.loc[ok_mask].copy()

required_cols = [
    "gvkey",
    "window_idx",
    "window_train_start",
    "window_train_end",
    "window_oos_start",
    "window_oos_end",
    "alpha",
    "beta1",
    "delta",
    "beta0",
]
missing = [c for c in required_cols if c not in freq_raw.columns]
if missing:
    raise ValueError(
        f"Missing required columns in nig_weekly_freq.csv: {missing}\n"
        f"Available columns: {list(freq_raw.columns)}"
    )

# sort first so 'last' is deterministic inside each window
freq_raw = freq_raw.sort_values(["gvkey", "window_idx", "date"]).reset_index(drop=True)

# collapse weekly OOS rows -> one row per firm-window
freq_init_df = (
    freq_raw.groupby(
        ["gvkey", "window_idx", "window_train_start", "window_train_end"],
        as_index=False,
        dropna=False,
    )
    .agg(
        train_start=("window_train_start", "first"),
        train_end=("window_train_end", "first"),
        oos_start=("window_oos_start", "first"),
        oos_end=("window_oos_end", "first"),
        alpha=("alpha", "last"),
        beta1=("beta1", "last"),
        delta=("delta", "last"),
        beta0=("beta0", "last"),
    )
)

# keep only complete usable rows
freq_init_df = freq_init_df.dropna(
    subset=["gvkey", "train_start", "train_end", "alpha", "beta1", "delta", "beta0"]
).copy()

freq_init_df = freq_init_df.sort_values(["gvkey", "train_end", "window_idx"]).reset_index(drop=True)

print("Using frequentist init file:", freq_file.name)
print("raw weekly rows:", len(freq_raw))
print("collapsed firm-window rows:", len(freq_init_df))
print("firms in init table:", freq_init_df["gvkey"].nunique())
print(
    "unique firm-window endpoints:",
    freq_init_df[["gvkey", "train_end"]].drop_duplicates().shape[0],
)

display(freq_init_df.head())


Using frequentist init file: nig_em_afonso.csv
raw weekly rows: 10623
collapsed firm-window rows: 814
firms in init table: 21
unique firm-window endpoints: 814


,gvkey,window_idx,window_train_start,window_train_end,train_start,train_end,oos_start,oos_end,alpha,beta1,delta,beta0
0,100022,0,2012-04-06,2014-03-28,2012-04-06,2014-03-28,2014-04-04,2014-06-27,271.811197,60.985402,1.425752,-0.274974
1,100022,1,2012-07-06,2014-06-27,2012-07-06,2014-06-27,2014-07-04,2014-09-26,475.107602,239.733287,1.603545,-0.862584
2,100022,2,2012-10-05,2014-09-26,2012-10-05,2014-09-26,2014-10-03,2014-12-26,288.152866,75.712857,1.416519,-0.333768
3,100022,3,2013-01-04,2014-12-26,2013-01-04,2014-12-26,2015-01-02,2015-03-27,297.622694,67.456628,1.590442,-0.339679
4,100022,4,2013-04-05,2015-03-27,2013-04-05,2015-03-27,2015-04-03,2015-06-26,248.905592,43.254451,1.296466,-0.138839


In [ ]:
# Rolling configuration
TRAIN_YEARS = 2
STEP_FREQ = "QE"
WEEK_FREQ = "W-FRI"

FORECAST_HORIZON_YEARS = 1.0
PD_HORIZON_YEARS = 1.0
ANN_FACTOR = 52.0

DATA_END = pd.Timestamp("2024-12-31")
LAST_TRAIN_END_CAL = DATA_END - pd.offsets.QuarterEnd(1)

# debug / runtime controls
MAX_FIRMS = 1
MAX_WINDOWS = 1

# Gibbs sampling controls
MAX_ITER = 20
BURN_IN = 5
THIN = 5
SEED = 123

# default prior-dispersion controls around the frequentist NIG estimates
DEFAULT_B0_DIAG = (50.0, 50.0)
PHI_PRIOR_VARIANCE = 100.0
OMEGA = 1e-3


def next_friday(ts):
    ts = pd.Timestamp(ts)
    days_ahead = (4 - ts.weekday()) % 7
    return ts + pd.Timedelta(days=days_ahead)


def prev_friday(ts):
    ts = pd.Timestamp(ts)
    days_back = (ts.weekday() - 4) % 7
    return ts - pd.Timedelta(days=days_back)


# one-time preprocessing for speed
panel = nig_df.copy()
panel["gvkey"] = panel["gvkey"].astype(str)
panel["date"] = pd.to_datetime(panel["date"])

needed_cols = ["gvkey", "date", "company", "market_cap", "L", "rf", "cds"]
panel = panel[[c for c in needed_cols if c in panel.columns]].copy()

for c in ["market_cap", "L", "rf", "cds"]:
    if c in panel.columns:
        panel[c] = pd.to_numeric(panel[c], errors="coerce")

panel = (
    panel.dropna(subset=["date", "market_cap", "L", "rf"])
         .query("market_cap > 0 and L > 0")
         .sort_values(["gvkey", "date"])
         .reset_index(drop=True)
)

# build per-firm daily panels once
firm_daily = {}
for gvkey, g in panel.groupby("gvkey", sort=False):
    g = (
        g.sort_values("date")
         .groupby("date", as_index=False)
         .last()
         .reset_index(drop=True)
    )
    firm_daily[gvkey] = g

gvkeys_all = sorted(firm_daily.keys())
if MAX_FIRMS is not None:
    gvkeys_all = gvkeys_all[:int(MAX_FIRMS)]

print("Firms loaded:", len(firm_daily), "| Firms in run:", len(gvkeys_all))
print("Panel date range:", panel["date"].min().date(), "to", panel["date"].max().date())

# build the rolling quarter schedule, then snap boundaries to Fridays
global_min_date = panel["date"].min()

earliest_end_cal = global_min_date + pd.DateOffset(years=TRAIN_YEARS) - pd.Timedelta(days=1)
train_end_cals = pd.date_range(start=earliest_end_cal, end=LAST_TRAIN_END_CAL, freq=STEP_FREQ)
train_end_cals = pd.to_datetime(train_end_cals)

if MAX_WINDOWS is not None:
    train_end_cals = train_end_cals[:int(MAX_WINDOWS)]

windows = []
for train_end_cal in train_end_cals:
    train_start_cal = train_end_cal - pd.DateOffset(years=TRAIN_YEARS) + pd.Timedelta(days=1)
    oos_start_cal = train_end_cal + pd.Timedelta(days=1)
    oos_end_cal = train_end_cal + pd.offsets.QuarterEnd(1)

    train_start = next_friday(train_start_cal)
    train_end = prev_friday(train_end_cal)
    oos_start = next_friday(oos_start_cal)
    oos_end = prev_friday(oos_end_cal)

    windows.append(
        {
            "window_idx": len(windows),
            "train_start": pd.Timestamp(train_start),
            "train_end": pd.Timestamp(train_end),
            "oos_start": pd.Timestamp(oos_start),
            "oos_end": pd.Timestamp(oos_end),
        }
    )

windows_df = pd.DataFrame(windows)

print("LAST_TRAIN_END calendar:", LAST_TRAIN_END_CAL.date())
print("Number of windows:", len(windows_df))
display(windows_df.head())


Firms loaded: 21 | Firms in run: 1
Panel date range: 2014-01-01 to 2025-12-19
LAST_TRAIN_END calendar: 2024-09-30
Number of windows: 1


,window_idx,train_start,train_end,oos_start,oos_end
0,0,2014-01-03,2015-12-25,2016-01-01,2016-03-25


In [ ]:
# Quick validation run: 1 firm x 1 window
# Pick a firm-window pair that is guaranteed to exist in the frequentist init table.

VALID_MAX_ITER = 20
VALID_BURN_IN = 10
VALID_THIN = 1

test_row = (
    freq_init_df[["gvkey", "train_start", "train_end", "oos_start", "oos_end", "window_idx"]]
    .dropna()
    .sort_values(["gvkey", "train_end", "window_idx"])
    .iloc[0]
)

gvkey_test = str(test_row["gvkey"])

window_test_df = pd.DataFrame(
    [{
        "window_idx": int(test_row["window_idx"]),
        "train_start": pd.Timestamp(test_row["train_start"]),
        "train_end": pd.Timestamp(test_row["train_end"]),
        "oos_start": pd.Timestamp(test_row["oos_start"]),
        "oos_end": pd.Timestamp(test_row["oos_end"]),
    }]
)

print("gvkey_test:", gvkey_test)
display(window_test_df)

params_test, oos_test, posterior_test = process_one_firm_bayesian_nig(
    firm_daily[gvkey_test],
    window_test_df,
    em_params_source=freq_init_df,
    gvkey_col="gvkey",
    input_frequency="daily",
    week_freq=WEEK_FREQ,
    date_col="date",
    equity_col="market_cap",
    debt_col="L",
    rf_col="rf",
    ann_factor=ANN_FACTOR,
    forecast_horizon_years=FORECAST_HORIZON_YEARS,
    pd_horizon_years=PD_HORIZON_YEARS,
    max_iter=VALID_MAX_ITER,
    burn_in=VALID_BURN_IN,
    thin=VALID_THIN,
    default_B0_diag=DEFAULT_B0_DIAG,
    phi_prior_variance=PHI_PRIOR_VARIANCE,
    omega=OMEGA,
    discounting="continuous",
    rng_seed=SEED,
)

print("params_test shape:", params_test.shape)
print("oos_test shape:", oos_test.shape)
print("posterior_test objects:", len(posterior_test))

display(params_test)
display(oos_test.head())

gvkey_test: 100022


,window_idx,train_start,train_end,oos_start,oos_end
0,0,2012-04-06,2014-03-28,2014-04-04,2014-06-27


params_test shape: (1, 28)
oos_test shape: (13, 24)
posterior_test objects: 1


,gvkey,train_start,train_end,window_idx,n_obs,alpha_lo,alpha_med,alpha_hi,alpha_mean,beta1_lo,...,beta0_med,beta0_hi,beta0_mean,n_keep_requested,n_keep_actual,n_rejects,n_fail_z,n_fail_candidate,ok,msg
0,100022,2012-04-06,2014-03-28,0,13,0.5948,1.031303,1.138861,0.892564,0.100138,...,-0.178224,-0.065544,-0.306883,10,10,9,0,9,True,ok


,date,gvkey,window_idx,rf_used,B_used,asset_lo,asset_med,asset_hi,asset_mean,asset_n_valid,...,PD_Q_n_valid,PD_P_lo,PD_P_med,PD_P_hi,PD_P_mean,PD_P_n_valid,train_start,train_end,oos_start,oos_end
0,2014-04-04,100022,0,0.001217,1.027250e+11,1.084602e+11,1.269951e+11,1.475359e+11,1.245634e+11,10,...,10,0.143884,0.503884,0.526549,0.435953,10,2012-04-06,2014-03-28,2014-04-04,2014-06-27
1,2014-04-11,100022,0,0.001092,1.027250e+11,1.055133e+11,1.237603e+11,1.449581e+11,1.215766e+11,10,...,10,0.152327,0.515335,0.541782,0.448044,10,2012-04-06,2014-03-28,2014-04-04,2014-06-27
2,2014-04-18,100022,0,0.000972,1.027250e+11,1.070722e+11,1.254812e+11,1.463280e+11,1.231620e+11,10,...,10,0.147766,0.509245,0.533665,0.441594,10,2012-04-06,2014-03-28,2014-04-04,2014-06-27
3,2014-04-25,100022,0,0.001185,1.027250e+11,1.049784e+11,1.231666e+11,1.444863e+11,1.210308e+11,10,...,10,0.153939,0.517441,0.544587,0.450281,10,2012-04-06,2014-03-28,2014-04-04,2014-06-27
4,2014-05-02,100022,0,0.000924,1.027250e+11,1.045566e+11,1.227122e+11,1.441261e+11,1.206087e+11,10,...,10,0.155184,0.519108,0.546725,0.452016,10,2012-04-06,2014-03-28,2014-04-04,2014-06-27


In [ ]:
# Full run: all firms x eligible windows, parallelized across firms

import os

# prevent BLAS oversubscription when joblib parallelizes across firms
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

N_CORES_AVAILABLE = os.cpu_count() or 1
N_JOBS = min(len(gvkeys_all), N_CORES_AVAILABLE)

print("Available CPU cores:", N_CORES_AVAILABLE)
print("Parallel jobs:", N_JOBS)


def _stable_firm_seed(base_seed: int, gvkey: str) -> int:
    s = str(gvkey)
    offset = sum((i + 1) * ord(ch) for i, ch in enumerate(s))
    return int((base_seed + offset) % (2**32 - 1))


def _run_one_firm_bayes_nig(gvkey: str):
    gvkey = str(gvkey)

    try:
        firm_df = firm_daily[gvkey].copy()

        em_firm = (
            freq_init_df.loc[freq_init_df["gvkey"].astype(str) == gvkey].copy()
            .sort_values(["train_end", "window_idx"])
            .reset_index(drop=True)
        )

        if em_firm.empty:
            fail = pd.DataFrame([{
                "gvkey": gvkey,
                "window_idx": np.nan,
                "train_start": pd.NaT,
                "train_end": pd.NaT,
                "ok": False,
                "msg": "no_frequentist_init_for_firm",
            }])
            return fail, pd.DataFrame(), []

        window_plan_firm = (
            windows_df.merge(
                em_firm[["train_start", "train_end"]].drop_duplicates(),
                on=["train_start", "train_end"],
                how="inner",
            )
            .sort_values("window_idx")
            .reset_index(drop=True)
        )

        if window_plan_firm.empty:
            fail = pd.DataFrame([{
                "gvkey": gvkey,
                "window_idx": np.nan,
                "train_start": pd.NaT,
                "train_end": pd.NaT,
                "ok": False,
                "msg": "no_matching_windows_between_schedule_and_freq_init",
            }])
            return fail, pd.DataFrame(), []

        firm_seed = _stable_firm_seed(SEED, gvkey)

        params_df_i, oos_df_i, posterior_i = process_one_firm_bayesian_nig(
            firm_df,
            window_plan_firm,
            em_params_source=freq_init_df,
            train_start_col="train_start",
            train_end_col="train_end",
            oos_start_col="oos_start",
            oos_end_col="oos_end",
            gvkey_col="gvkey",
            input_frequency="daily",
            week_freq=WEEK_FREQ,
            date_col="date",
            equity_col="market_cap",
            debt_col="L",
            rf_col="rf",
            ann_factor=ANN_FACTOR,
            forecast_horizon_years=FORECAST_HORIZON_YEARS,
            pd_horizon_years=PD_HORIZON_YEARS,
            max_iter=MAX_ITER,
            burn_in=BURN_IN,
            thin=THIN,
            prior_b0=None,
            prior_B0=None,
            prior_hyper=None,
            default_B0_diag=DEFAULT_B0_DIAG,
            phi_prior_variance=PHI_PRIOR_VARIANCE,
            omega=OMEGA,
            discounting="continuous",
            rng_seed=firm_seed,
        )

        return params_df_i, oos_df_i, posterior_i

    except Exception as exc:
        fail = pd.DataFrame([{
            "gvkey": gvkey,
            "window_idx": np.nan,
            "train_start": pd.NaT,
            "train_end": pd.NaT,
            "ok": False,
            "msg": f"firm_job_fail:{type(exc).__name__}:{str(exc)[:250]}",
        }])
        return fail, pd.DataFrame(), []


parallel_results = Parallel(
    n_jobs=N_JOBS,
    backend="loky",
    verbose=10,
)(
    delayed(_run_one_firm_bayes_nig)(gvkey)
    for gvkey in gvkeys_all
)

params_parts = []
oos_parts = []
posterior_all = []

for params_df_i, oos_df_i, posterior_i in parallel_results:
    if params_df_i is not None and not params_df_i.empty:
        params_parts.append(params_df_i.copy())
    if oos_df_i is not None and not oos_df_i.empty:
        oos_parts.append(oos_df_i.copy())
    if posterior_i:
        posterior_all.extend(posterior_i)

params_all = (
    pd.concat(params_parts, ignore_index=True)
    .sort_values(["gvkey", "window_idx", "train_end"], na_position="last")
    .reset_index(drop=True)
    if params_parts else pd.DataFrame()
)

oos_all = (
    pd.concat(oos_parts, ignore_index=True)
    .sort_values(["gvkey", "window_idx", "date"], na_position="last")
    .reset_index(drop=True)
    if oos_parts else pd.DataFrame()
)

print("params_all shape:", params_all.shape)
print("oos_all shape:", oos_all.shape)
print("posterior objects:", len(posterior_all))

if not params_all.empty and "ok" in params_all.columns:
    ok_mask = params_all["ok"].astype(str).str.lower().isin(["true", "1", "yes"])
    print("successful windows:", int(ok_mask.sum()), "/", len(params_all))

display(params_all.head())
display(oos_all.head())

Available CPU cores: 12
Parallel jobs: 1
params_all shape: (1, 28)
oos_all shape: (13, 24)
posterior objects: 1
successful windows: 1 / 1


[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    4.0s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    4.0s finished


,gvkey,train_start,train_end,window_idx,n_obs,alpha_lo,alpha_med,alpha_hi,alpha_mean,beta1_lo,...,beta0_med,beta0_hi,beta0_mean,n_keep_requested,n_keep_actual,n_rejects,n_fail_z,n_fail_candidate,ok,msg
0,100022,2014-01-03,2015-12-25,0,104,6816.188225,6850.340523,6851.779692,6838.224549,6189.300699,...,-8.046223,-8.023322,-8.044052,3,3,0,0,0,True,ok


,date,gvkey,window_idx,rf_used,B_used,asset_lo,asset_med,asset_hi,asset_mean,asset_n_valid,...,PD_Q_n_valid,PD_P_lo,PD_P_med,PD_P_hi,PD_P_mean,PD_P_n_valid,train_start,train_end,oos_start,oos_end
0,2016-01-01,100022,0,-0.003968,1.173660e+11,1.766054e+11,1.766054e+11,1.766054e+11,1.766054e+11,3,...,3,1.037451e-09,2.501919e-09,1.178794e-08,5.246324e-09,3,2014-01-03,2015-12-25,2016-01-01,2016-03-25
1,2016-01-08,100022,0,-0.004354,1.173660e+11,1.681086e+11,1.681086e+11,1.681086e+11,1.681086e+11,3,...,3,3.588259e-08,7.591376e-08,3.286740e-07,1.505555e-07,3,2014-01-03,2015-12-25,2016-01-01,2016-03-25
2,2016-01-15,100022,0,-0.004190,1.173660e+11,1.648264e+11,1.648264e+11,1.648265e+11,1.648264e+11,3,...,3,1.338510e-07,2.692001e-07,1.119448e-06,5.200417e-07,3,2014-01-03,2015-12-25,2016-01-01,2016-03-25
3,2016-01-22,100022,0,-0.004325,1.173660e+11,1.658597e+11,1.658597e+11,1.658597e+11,1.658597e+11,3,...,3,8.875494e-08,1.813585e-07,7.640334e-07,3.533134e-07,3,2014-01-03,2015-12-25,2016-01-01,2016-03-25
4,2016-01-29,100022,0,-0.004508,1.173660e+11,1.640451e+11,1.640451e+11,1.640452e+11,1.640451e+11,3,...,3,1.822036e-07,3.620918e-07,1.490724e-06,6.949843e-07,3,2014-01-03,2015-12-25,2016-01-01,2016-03-25


In [ ]:
# Build MCMC diagnostics table across all firms and windows

mcmc_diag_all = build_mcmc_diagnostics_df(posterior_all)

print("mcmc_diag_all shape:", mcmc_diag_all.shape)
display(mcmc_diag_all.head(20))

mcmc_diag_all shape: (4, 5)


,gvkey,window_idx,param,ac_1,geweke_z
0,100022,0,alpha,-1.0,NaN
1,100022,0,beta0,-1.0,NaN
2,100022,0,beta1,-1.0,NaN
3,100022,0,delta,-1.0,NaN


In [ ]:
# Save outputs
bayes_nig_out = output_path / "bayesian_nig"
bayes_nig_out.mkdir(parents=True, exist_ok=True)

run_tag = (
    f"condfixA_iter{MAX_ITER}_burn{BURN_IN}_thin{THIN}_"
    f"firms{len(gvkeys_all)}_wins{len(windows_df)}"
)

params_file = bayes_nig_out / f"bayes_nig_params_{run_tag}.csv"
oos_file = bayes_nig_out / f"bayes_nig_oos_weekly_{run_tag}.csv"
windows_file = bayes_nig_out / f"bayes_nig_windows_{run_tag}.csv"
settings_file = bayes_nig_out / f"bayes_nig_run_settings_{run_tag}.csv"
posterior_file = bayes_nig_out / f"bayes_nig_posterior_{run_tag}.joblib"
mcmc_diag_file = bayes_nig_out / f"bayes_nig_mcmc_diag_{run_tag}.csv"


if not params_all.empty:
    params_all.to_csv(params_file, index=False)

if not oos_all.empty:
    oos_all.to_csv(oos_file, index=False)

windows_df.to_csv(windows_file, index=False)

if not mcmc_diag_all.empty:
    mcmc_diag_all.to_csv(mcmc_diag_file, index=False)

run_settings = pd.DataFrame(
    [
        {
            "train_years": TRAIN_YEARS,
            "step_freq": STEP_FREQ,
            "week_freq": WEEK_FREQ,
            "forecast_horizon_years": FORECAST_HORIZON_YEARS,
            "pd_horizon_years": PD_HORIZON_YEARS,
            "ann_factor": ANN_FACTOR,
            "data_end": DATA_END,
            "last_train_end_cal": LAST_TRAIN_END_CAL,
            "max_firms": MAX_FIRMS,
            "max_windows": MAX_WINDOWS,
            "max_iter": MAX_ITER,
            "burn_in": BURN_IN,
            "thin": THIN,
            "seed": SEED,
            "default_B0_diag_1": DEFAULT_B0_DIAG[0],
            "default_B0_diag_2": DEFAULT_B0_DIAG[1],
            "phi_prior_variance": PHI_PRIOR_VARIANCE,
            "omega": OMEGA,
            "n_jobs": N_JOBS,
            "n_firms_run": len(gvkeys_all),
            "n_windows_schedule": len(windows_df),
            "n_params_rows": len(params_all),
            "n_oos_rows": len(oos_all),
            "n_posterior_objects": len(posterior_all),
        }
    ]
)
run_settings.to_csv(settings_file, index=False)

dump(posterior_all, posterior_file, compress=3)

print("Saved files:")
for fp in [params_file, oos_file, windows_file, settings_file, posterior_file, mcmc_diag_file]:
    print(" -", fp)

Saved files:
 - c:\Users\afons\OneDrive\Desktop\ESE\FCS\Merton_NIGbayesian\notebooks_test\..\data\derived\bayesian_nig\bayes_nig_params_condfixA_iter20_burn5_thin5_firms1_wins1.csv
 - c:\Users\afons\OneDrive\Desktop\ESE\FCS\Merton_NIGbayesian\notebooks_test\..\data\derived\bayesian_nig\bayes_nig_oos_weekly_condfixA_iter20_burn5_thin5_firms1_wins1.csv
 - c:\Users\afons\OneDrive\Desktop\ESE\FCS\Merton_NIGbayesian\notebooks_test\..\data\derived\bayesian_nig\bayes_nig_windows_condfixA_iter20_burn5_thin5_firms1_wins1.csv
 - c:\Users\afons\OneDrive\Desktop\ESE\FCS\Merton_NIGbayesian\notebooks_test\..\data\derived\bayesian_nig\bayes_nig_run_settings_condfixA_iter20_burn5_thin5_firms1_wins1.csv
 - c:\Users\afons\OneDrive\Desktop\ESE\FCS\Merton_NIGbayesian\notebooks_test\..\data\derived\bayesian_nig\bayes_nig_posterior_condfixA_iter20_burn5_thin5_firms1_wins1.joblib
 - c:\Users\afons\OneDrive\Desktop\ESE\FCS\Merton_NIGbayesian\notebooks_test\..\data\derived\bayesian_nig\bayes_nig_mcmc_diag_cond

In [ ]:
# Quick diagnostics

if not params_all.empty:
    cols_show = [c for c in [
        "gvkey",
        "window_idx",
        "train_start",
        "train_end",
        "ok",
        "msg",
        "n_obs",
        "n_keep_requested",
        "n_keep_actual",
        "n_rejects",
        "n_fail_z",
        "n_fail_candidate",
        "alpha_med",
        "beta1_med",
        "delta_med",
        "beta0_med",
    ] if c in params_all.columns]

    display(params_all[cols_show].head(20))

    if "ok" in params_all.columns:
        fail_mask = ~params_all["ok"].astype(str).str.lower().isin(["true", "1", "yes"])
        if fail_mask.any():
            print("Most common failure messages:")
            display(params_all.loc[fail_mask, "msg"].value_counts().head(20))

if not oos_all.empty:
    cols_show_oos = [c for c in [
        "gvkey",
        "window_idx",
        "date",
        "rf_used",
        "B_used",
        "asset_med",
        "PD_P_med",
        "PD_Q_med",
    ] if c in oos_all.columns]
    display(oos_all[cols_show_oos].head(20))

if posterior_all:
    diag0 = summarize_chain_diagnostics(
        np.asarray(posterior_all[0]["result"]["params_draws_annual"], dtype=float)
    )
    print("Single-chain diagnostics for first posterior object:")
    display(diag0)

,gvkey,window_idx,train_start,train_end,ok,msg,n_obs,n_keep_requested,n_keep_actual,n_rejects,n_fail_z,n_fail_candidate,alpha_med,beta1_med,delta_med,beta0_med
0,100022,0,2014-01-03,2015-12-25,True,ok,104,3,3,0,0,0,6850.340523,6190.818978,3.836212,-8.046223


,gvkey,window_idx,date,rf_used,B_used,asset_med,PD_P_med,PD_Q_med
0,100022,0,2016-01-01,-0.003968,1.173660e+11,1.766054e+11,2.501919e-09,5.308564e-07
1,100022,0,2016-01-08,-0.004354,1.173660e+11,1.681086e+11,7.591376e-08,1.004294e-05
2,100022,0,2016-01-15,-0.004190,1.173660e+11,1.648264e+11,2.692001e-07,2.882201e-05
3,100022,0,2016-01-22,-0.004325,1.173660e+11,1.658597e+11,1.813585e-07,2.085227e-05
4,100022,0,2016-01-29,-0.004508,1.173660e+11,1.640451e+11,3.620918e-07,3.748960e-05
5,100022,0,2016-02-05,-0.004592,1.173660e+11,1.614965e+11,9.394699e-07,8.296916e-05
6,100022,0,2016-02-12,-0.004854,1.173660e+11,1.602330e+11,1.495264e-06,1.231139e-04
7,100022,0,2016-02-19,-0.004904,1.173660e+11,1.623881e+11,6.746475e-07,6.404717e-05
8,100022,0,2016-02-26,-0.005040,1.173660e+11,1.624403e+11,6.616466e-07,6.343487e-05
9,100022,0,2016-03-04,-0.005090,1.173660e+11,1.677078e+11,8.876017e-08,1.190462e-05


Single-chain diagnostics for first posterior object:


,param,n_draws,mean,sd,geweke_z,lag1_acf
0,alpha,3,6838.224549,22.384607,NaN,-1.0
1,beta1,3,6192.701786,4.796827,NaN,-1.0
2,delta,3,3.802573,0.064363,NaN,-1.0
3,beta0,3,-8.044052,0.022268,NaN,-1.0


In [ ]:
# Optional: inspect the worst windows by rejection count

if not params_all.empty and "n_rejects" in params_all.columns:
    worst = (
        params_all.sort_values("n_rejects", ascending=False)
        .head(10)
        .reset_index(drop=True)
    )
    display(worst)

,gvkey,train_start,train_end,window_idx,n_obs,alpha_lo,alpha_med,alpha_hi,alpha_mean,beta1_lo,...,beta0_med,beta0_hi,beta0_mean,n_keep_requested,n_keep_actual,n_rejects,n_fail_z,n_fail_candidate,ok,msg
0,100022,2014-01-03,2015-12-25,0,104,6816.188225,6850.340523,6851.779692,6838.224549,6189.300699,...,-8.046223,-8.023322,-8.044052,3,3,0,0,0,True,ok


In [ ]:
# Optional: inspect diagnostics for a chosen firm-window

gvkey_pick = str(params_all.loc[0, "gvkey"])
window_pick = int(params_all.loc[0, "window_idx"])

matches = [
    p for p in posterior_all
    if str(p["gvkey"]) == gvkey_pick and int(p["window_idx"]) == window_pick
]

if matches:
    diag_pick = summarize_chain_diagnostics(
        np.asarray(matches[0]["result"]["params_draws_annual"], dtype=float)
    )
    display(diag_pick)
else:
    print("No posterior object found for that firm-window.")

,param,n_draws,mean,sd,geweke_z,lag1_acf
0,alpha,3,6838.224549,22.384607,NaN,-1.0
1,beta1,3,6192.701786,4.796827,NaN,-1.0
2,delta,3,3.802573,0.064363,NaN,-1.0
3,beta0,3,-8.044052,0.022268,NaN,-1.0


In [9]:
FILE_NAME = "bayes_nig_mcmc_diag_condfixA_iter4000_burn500_thin5_firms21_wins36.csv"
RELATIVE_FOLDER = Path("data/derived/bayesian_nig")
Z_THRESHOLD = 2.0          # flag a parameter if |geweke_z| > Z_THRESHOLD
MAX_BAD_PARAMS_OK = 2      # 0 or 1 bad parameter = OK; >1 = flagged window


def resolve_csv_path(file_name: str, relative_folder: Path) -> Path:
    candidates = [
        Path.cwd() / relative_folder / file_name,
        Path.cwd().parent / relative_folder / file_name,
        Path.cwd().parent.parent / relative_folder / file_name,
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(
        f"Could not find '{file_name}' in '{relative_folder}'.\n"
        f"Tried:\n" + "\n".join(str(p) for p in candidates)
    )


csv_path = resolve_csv_path(FILE_NAME, RELATIVE_FOLDER)
diag = pd.read_csv(csv_path)

DROP_GVKEYS = {
    "241456",
    "101336",
    "17436",
    "201794",
    "220940",
    "100022",
    "221616",
    "61616",
    "101202",
}

if "gvkey" in diag.columns:
    diag["gvkey"] = diag["gvkey"].astype(str)
    diag = diag.loc[~diag["gvkey"].isin(DROP_GVKEYS)].copy()

print("Loaded file:", csv_path)
print("Shape:", diag.shape)
print("Columns:", list(diag.columns))


required_cols = {"gvkey", "window_idx", "param", "geweke_z"}
missing = required_cols - set(diag.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

diag = diag.copy()
diag["gvkey"] = diag["gvkey"].astype(str)

# Parameter-level indicator: does this param fail the Geweke threshold?
diag["bad_geweke"] = diag["geweke_z"].abs() > Z_THRESHOLD


bad_counts = (
    diag.loc[diag["bad_geweke"]]
    .groupby(["gvkey", "window_idx"])["param"]
    .nunique()
    .rename("n_bad_params")
    .reset_index()
)

all_windows = (
    diag[["gvkey", "window_idx"]]
    .drop_duplicates()
    .sort_values(["gvkey", "window_idx"])
    .reset_index(drop=True)
)

window_summary = all_windows.merge(
    bad_counts,
    on=["gvkey", "window_idx"],
    how="left"
)

window_summary["n_bad_params"] = window_summary["n_bad_params"].fillna(0).astype(int)
window_summary["flagged_window"] = window_summary["n_bad_params"] > MAX_BAD_PARAMS_OK


firm_summary = (
    window_summary
    .groupby("gvkey", as_index=False)
    .agg(
        n_windows=("window_idx", "nunique"),
        n_flagged_windows=("flagged_window", "sum"),
    )
)

firm_summary["flagged_fraction"] = (
    firm_summary["n_flagged_windows"] / firm_summary["n_windows"]
)

firm_summary = firm_summary.sort_values(
    ["flagged_fraction", "n_flagged_windows", "gvkey"],
    ascending=[False, False, True]
).reset_index(drop=True)


overall_summary = pd.DataFrame({
    "n_firms": [window_summary["gvkey"].nunique()],
    "n_windows_total": [len(window_summary)],
    "n_flagged_windows_total": [int(window_summary["flagged_window"].sum())],
})

overall_summary["flagged_fraction_total"] = (
    overall_summary["n_flagged_windows_total"] / overall_summary["n_windows_total"]
)


print("\n=== Window-level summary (first 10 rows) ===")
display(window_summary.head(10))

print("\n=== Firm-level summary ===")
display(firm_summary)

print("\n=== Overall summary ===")
display(overall_summary)

Loaded file: c:\Users\afons\OneDrive\Desktop\ESE\FCS\Merton_NIGbayesian\data\derived\bayesian_nig\bayes_nig_mcmc_diag_condfixA_iter4000_burn500_thin5_firms21_wins36.csv
Shape: (1728, 5)
Columns: ['gvkey', 'window_idx', 'param', 'ac_1', 'geweke_z']

=== Window-level summary (first 10 rows) ===


,gvkey,window_idx,n_bad_params,flagged_window
0,100080,0,3,True
1,100080,1,2,False
2,100080,2,2,False
3,100080,3,2,False
4,100080,4,0,False
5,100080,5,3,True
6,100080,6,0,False
7,100080,7,2,False
8,100080,8,0,False
9,100080,9,1,False



=== Firm-level summary ===


,gvkey,n_windows,n_flagged_windows,flagged_fraction
0,222379,36,7,0.194444
1,100957,36,6,0.166667
2,101361,36,6,0.166667
3,19349,36,6,0.166667
4,221244,36,6,0.166667
5,101204,36,5,0.138889
6,102296,36,4,0.111111
7,100080,36,3,0.083333
8,17452,36,3,0.083333
9,14447,36,2,0.055556



=== Overall summary ===


,n_firms,n_windows_total,n_flagged_windows_total,flagged_fraction_total
0,12,432,51,0.118056
